# Specimen 02 — Titanic Survival Prediction

Goal: feature engineering and honest evaluation on messy, real-world data.

In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score


## 1. Load & audit

`sns.load_dataset('titanic')` or the Kaggle CSV. Tabulate missingness per column.

In [36]:
# Ssl is a module that provides access to Transport Layer Security (previously and widely known as Secure Sockets Layer) encryption and peer authentication facilities for network sockets, both client-side and server-side.
import ssl
# This is to fix the SSL certificate verification issue when loading datasets from seaborn
import certifi
# This line of code sets the default SSL context to an unverified context, which means that SSL certificate verification will be disabled. This is often done to bypass SSL certificate verification errors when making HTTPS requests, especially in environments where the SSL certificates may not be properly configured or trusted.
ssl._create_default_https_context = ssl._create_unverified_context

import seaborn as sns
# Load the Titanic dataset from seaborn
df = sns.load_dataset('titanic')
# Display the first few rows of the dataset to understand its structure and contents
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## 2. EDA by group

Survival rate by sex, passenger class, and age band.

In [37]:
# Survival rate by sex
print("Survival rate by sex:")
# Group the dataset by 'sex' and calculate the mean survival rate for each group, rounding the results to three decimal places for better readability
print(df.groupby('sex')['survived'].mean().round(3))

# Survival rate by passenger class
print("\nSurvival rate by class:")
# Group the dataset by 'pclass' (passenger class) and calculate the mean survival rate for each class, rounding the results to three decimal places for better readability
print(df.groupby('pclass')['survived'].mean().round(3))

# Survival rate by age band
df['age_band'] = pd.cut(df['age'], bins=[0, 12, 18, 35, 60, 100],
    labels=['Child', 'Teen', 'Adult', 'Middle Age', 'Senior'])
# Group the dataset by 'age_band' and calculate the mean survival rate for each age band, rounding the results to three decimal places for better readability
print("\nSurvival rate by age band:")
# Group the dataset by 'age_band' and calculate the mean survival rate for each age band, rounding the results to three decimal places for better readability
print(df.groupby('age_band', observed=True)['survived'].mean().round(3))

Survival rate by sex:
sex
female    0.742
male      0.189
Name: survived, dtype: float64

Survival rate by class:
pclass
1    0.630
2    0.473
3    0.242
Name: survived, dtype: float64

Survival rate by age band:
age_band
Child         0.580
Teen          0.429
Adult         0.383
Middle Age    0.400
Senior        0.227
Name: survived, dtype: float64


## 3. Engineer 2+ features

e.g. `Title` from name, `FamilySize` from siblings + parents. Justify each.

In [38]:
# Survival rate by family size, which is calculated as the sum of siblings/spouses (sibsp) and parents/children (parch) plus one (the passenger themselves)
df['family_size'] = df['sibsp'] + df['parch'] + 1

# Create a new column 'is_alone' to indicate whether a passenger is traveling alone (1) or with family (0)
df['is_alone'] = (df['family_size'] == 1).astype(int)

# Display the first few rows of the relevant columns to verify the calculations
print(df[['sibsp', 'parch', 'family_size', 'is_alone']].head())

#Survival rate by family size
print("\nSurvival rate by family size:")
# Group the dataset by 'family_size' and calculate the mean survival rate for each family size, rounding the results to three decimal places for better readability
print(df.groupby('family_size')['survived'].mean().round(3))

#Survival rate by is_alone
print("\nSurvival rate by is_alone:")
# Group the dataset by 'is_alone' and calculate the mean survival rate for passengers traveling alone versus those traveling with family, rounding the results to three decimal places for better readability
print(df.groupby('is_alone')['survived'].mean().round(3))

   sibsp  parch  family_size  is_alone
0      1      0            2         0
1      1      0            2         0
2      0      0            1         1
3      1      0            2         0
4      0      0            1         1

Survival rate by family size:
family_size
1     0.304
2     0.553
3     0.578
4     0.724
5     0.200
6     0.136
7     0.333
8     0.000
11    0.000
Name: survived, dtype: float64

Survival rate by is_alone:
is_alone
0    0.506
1    0.304
Name: survived, dtype: float64


## 4. Preprocessing pipeline

`ColumnTransformer` with `SimpleImputer` + `OneHotEncoder`. Fit on training folds only.

In [39]:
#Compose a preprocessing pipeline for the Titanic dataset, which includes both numeric and categorical features. The pipeline will handle missing values, scale numeric features, and one-hot encode categorical features.
#Import ColumnTransformer is a class in scikit-learn that allows you to apply different preprocessing steps to different subsets of features in your dataset. It is particularly useful when you have a mix of numeric and categorical features that require different types of preprocessing.
from sklearn.compose import ColumnTransformer
#Pipeline module is a class in scikit-learn that allows you to create a sequence of data transformation and modeling steps that can be treated as a single object. It helps streamline the process of applying multiple transformations and fitting a model, making it easier to manage and reproduce your machine learning workflow.
#Import Pipeline from sklearn.pipeline module. The Pipeline class allows you to create a sequence of data transformation and modeling steps that can be treated as a single object. It helps streamline the process of applying multiple transformations and fitting a model, making it easier to manage and reproduce your machine learning workflow.
from sklearn.pipeline import Pipeline
#Imputer is a class in scikit-learn that provides strategies for handling missing values in your dataset. It allows you to fill in missing values using various techniques, such as replacing them with the mean, median, or most frequent value of the feature.
#Import SimpleImputer is a specific implementation of the Imputer class that provides simple strategies for imputing missing values. It can be used to fill in missing values with a constant value, the mean, median, or most frequent value of the feature.
from sklearn.impute import SimpleImputer
#Preprocessing module in scikit-learn provides various tools for transforming and scaling data before feeding it into machine learning models. It includes classes for encoding categorical variables, scaling numeric features, and handling missing values.
#OneHotEncoder is a class in scikit-learn that converts categorical variables into a binary matrix representation, where each category is represented by a separate column with a value of 1 or 0 indicating the presence or absence of that category.
#StandardScaler is a class in scikit-learn that standardizes features by removing the mean and scaling to unit variance. It transforms the data to have a mean of 0 and a standard
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Define the numeric and categorical features to be used in the preprocessing pipeline
numeric_features = ['age', 'fare', 'sibsp', 'parch', 'family_size']
categorical_features = ['pclass', 'sex', 'embarked', 'is_alone']

# Split the dataset into training and test sets, with 80% of the data used for training and 20% for testing. The split is stratified based on the target variable 'survived' to ensure that both sets have a similar distribution of survival outcomes. A random seed is set for reproducibility.
X = df[numeric_features + categorical_features]
# The target variable 'survived' is extracted from the dataset to be used for training and evaluation of the machine learning models.
y = df['survived']

# Split the dataset into training and test sets, with 80% of the data used for training and 20% for testing. The split is stratified based on the target variable 'survived' to ensure that both sets have a similar distribution of survival outcomes. A random seed is set for reproducibility.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Numeric pipeline: fill missing values (mostly 'age' and 'fare') with the median, then scale to zero mean and unit variance
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
# Categorical pipeline: fill missing values (mostly 'embarked') with the most frequent value, then one-hot encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
# Compose a preprocessing pipeline that applies the numeric and categorical transformers to their respective feature sets. The ColumnTransformer allows you to specify which transformer to apply to which subset of features, enabling you to handle both numeric and categorical data in a single pipeline.
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])
# Apply the preprocessing pipeline to the training and test sets, transforming the features according to the specified steps in the numeric and categorical pipelines. The transformed data is returned as NumPy arrays, which can be used for training and evaluating machine learning models.
print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Training set: (712, 9)
Test set: (179, 9)


## 5. Two naive baselines

'Everyone dies' and 'all women survive, all men die' accuracy.

In [40]:
#Metrics module in scikit-learn provides functions for evaluating the performance of machine learning models. It includes various metrics for classification, regression, and clustering tasks, allowing you to assess how well your models are performing.
#Accuracy score is a metric that measures the proportion of correctly predicted instances out of the total instances in a classification problem. It is calculated as the number of correct predictions divided by the total number of predictions, and it provides a simple way to evaluate the overall performance of a classification model.
from sklearn.metrics import accuracy_score
# Baseline 1: predict all passengers die.
baseline_all_die = pd.Series(0, index=y_test.index)
#Accuracy score is calculated by comparing the predicted values (all zeros) with the actual values in the test set. The accuracy score is then printed to provide a reference point for evaluating the performance of more complex models.
acc_all_die = accuracy_score(y_test, baseline_all_die)
# This line of code calculates the accuracy of the baseline model that predicts all passengers die (the majority class) by comparing the predicted values (all zeros) with the actual values in the test set. The accuracy score is then printed to provide a reference point for evaluating the performance of more complex models.
print(f"'Everyone dies' baseline accuracy: {acc_all_die:.3f}")

# Baseline 2: predict all women survive, all men die.
baseline_sex_rule = (X_test['sex'] == 'female').astype(int)
#Accuracy score is calculated by comparing the predicted values (based on the sex rule) with the actual values in the test set. The accuracy score is then printed to provide a reference point for evaluating the performance of more complex models.
acc_sex_rule = accuracy_score(y_test, baseline_sex_rule)
# This line of code calculates the accuracy of the baseline model that predicts all women survive and all men die by comparing the predicted values with the actual values in the test set. The accuracy score is then printed to provide a reference point for evaluating the performance of more complex models.
print(f"'Women survive, men die' baseline accuracy: {acc_sex_rule:.3f}")

# Any real model needs to clear both of these bars to justify its complexity

'Everyone dies' baseline accuracy: 0.615
'Women survive, men die' baseline accuracy: 0.777


## 6. Compare model families

Logistic regression, random forest, gradient boosting — stratified CV.

In [ ]:
# Linear models are a class of machine learning algorithms that assume a linear relationship between the input features and the target variable. They are simple, interpretable, and often perform well on linearly separable data. Examples include Linear Regression for regression tasks and Logistic Regression for binary classification tasks.
from sklearn.linear_model import LogisticRegression
#Enemble models are a class of machine learning algorithms that combine multiple individual models (often called "weak learners") to create a stronger overall model. The idea is that by aggregating the predictions of several models, the ensemble can achieve better performance and generalization than any single model alone. Examples of ensemble methods include Random Forests, Gradient Boosting, and AdaBoost.
# Random Forest is an ensemble learning method that constructs multiple decision trees during training and outputs the mode of the classes (classification) or mean prediction (regression) of the individual trees. It helps improve accuracy and control overfitting.
# Gradient Boosting is an ensemble learning technique that builds a series of weak learners (typically decision trees) in a sequential manner, where each subsequent model attempts to correct the errors of the previous models
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

# 5-fold stratified CV keeps the survived/died ratio consistent across every fold, which matters here since survival is imbalanced (~38% survived)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Evaluate each model using cross-validation and store the results in a dictionary. The cross_val_score function performs the cross-validation, fitting the model on the training folds and evaluating it on the validation fold. The accuracy scores for each fold are collected, and the mean and standard deviation of the scores are printed for each model.
cv_results = {}
for name, model in models.items():
    # Each pipeline refits its own preprocessor on every training fold, so imputation/scaling/encoding statistics never see the validation fold
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    # Perform cross-validation on the pipeline using the training data (X_train, y_train) and the specified cross-validation strategy (cv). The scoring parameter is set to 'accuracy' to evaluate the models based on their accuracy scores. The resulting accuracy scores for each fold are stored in the cv_results dictionary under the corresponding model name.
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy')
    # Store the cross-validation scores for the current model in the cv_results dictionary, using the model name as the key. The mean and standard deviation of the accuracy scores across the folds are then printed to provide an overview of the model's performance and variability.
    cv_results[name] = scores
    # Print the mean and standard deviation of the accuracy scores for the current model, formatted to three decimal places. This provides a summary of the model's performance across the cross-validation folds, allowing for easy comparison between different models.
    print(f"{name}: {scores.mean():.3f} +/- {scores.std():.3f}")

Logistic Regression: 0.796 +/- 0.015
Random Forest: 0.794 +/- 0.017
Gradient Boosting: 0.823 +/- 0.034


## 7. Evaluate beyond accuracy

Precision/recall/F1 for the 'survived' class specifically.

In [ ]:
# Metrics module in scikit-learn provides functions for evaluating the performance of machine learning models. It includes various metrics for classification, regression, and clustering tasks, allowing you to assess how well your models are performing.
# Classification report is a function in scikit-learn that generates a text summary of the precision, recall, F1-score, and support for each class in a classification problem. It provides a comprehensive overview of the model's performance across different classes, allowing you to assess how well the model is performing for each class and identify any potential issues or imbalances in the predictions.
# Precision, recall, and F1-score are important metrics for evaluating classification models, especially in imbalanced datasets. Precision measures the proportion of true positive predictions among all positive predictions, recall measures the proportion of true positive predictions among all actual positives, and F1-score is the harmonic mean of precision and recall, providing a single metric that balances both aspects.
from sklearn.metrics import classification_report, precision_recall_fscore_support

# Refit each model family on the full training set, then evaluate once on the held-out test set (accuracy alone can hide a model that's great at spotting deaths but bad at spotting survivors, or vice versa)
last_pred = None
# Pipelines are used to streamline the process of applying multiple transformations and fitting a model, making it easier to manage and reproduce the machine learning workflow. In this case, each model is refitted on the full training set using a pipeline that includes the preprocessing steps and the classifier. The predictions are then made on the held-out test set, and precision, recall, and F1-score are calculated for the positive class (survived) to evaluate the model's performance.
for name, model in models.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    # Fit the pipeline on the full training set (X_train, y_train) and make predictions on the held-out test set (X_test). The predicted values are stored in the variable y_pred, and the last model's predictions are saved in last_pred for later evaluation.
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    last_pred = y_pred
    # Calculate precision, recall, and F1-score for the positive class (survived) using the precision_recall_fscore_support function. The average parameter is set to 'binary' to compute metrics for the positive class only, and pos_label is set to 1 to indicate that the positive class corresponds to passengers who survived. The results are printed for each model.
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average='binary', pos_label=1
    )
    # Print the precision, recall, and F1-score for the positive class (survived) for the current model, formatted to three decimal places. This provides a summary of the model's performance in terms of correctly identifying survivors, allowing for easy comparison between different models.
    print(f"--- {name} ---")
    print(f"Precision (survived): {precision:.3f}")
    print(f"Recall (survived):    {recall:.3f}")
    print(f"F1 (survived):        {f1:.3f}\n")

# Full report (precision/recall/F1 for both classes) for the last model above
print(classification_report(y_test, last_pred, target_names=['Died', 'Survived']))

--- Logistic Regression ---
Precision (survived): 0.810
Recall (survived):    0.681
F1 (survived):        0.740

--- Random Forest ---
Precision (survived): 0.790
Recall (survived):    0.710
F1 (survived):        0.748

--- Gradient Boosting ---
Precision (survived): 0.833
Recall (survived):    0.652
F1 (survived):        0.732

              precision    recall  f1-score   support

        Died       0.81      0.92      0.86       110
    Survived       0.83      0.65      0.73        69

    accuracy                           0.82       179
   macro avg       0.82      0.79      0.80       179
weighted avg       0.82      0.82      0.81       179



## 8. Feature importance check

Does it line up with your EDA findings from step 2?

In [ ]:
# Feature importances for the Random Forest model, which can help identify which features are most influential in predicting survival. The feature importances are extracted from the fitted Random Forest model and displayed in descending order.
rf_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=200, random_state=42))
])
# Fit a Random Forest on the training set to inspect which features it relies on
rf_pipe.fit(X_train, y_train)

# Feature names after preprocessing one-hot encoding expands each categorical column into multiple binary columns
feature_names = rf_pipe.named_steps['preprocessor'].get_feature_names_out()
importances = rf_pipe.named_steps['classifier'].feature_importances_

# Create a pandas Series to hold the feature importances, using the feature names as the index. The Series is then sorted in descending order to display the most important features first. The rounded values of the importances are printed to provide a clear overview of which features have the greatest impact on the model's predictions.
importance_series = pd.Series(importances, index=feature_names).sort_values(ascending=False)
print("Feature importances (Random Forest):")
print(importance_series.round(3))

# Compare against the step 2 EDA: sex and pclass showed the largest
# survival-rate gaps there, so we'd expect them to dominate the importances too

Feature importances (Random Forest):
num__fare           0.244
num__age            0.240
cat__sex_male       0.140
cat__sex_female     0.136
cat__pclass_3       0.048
num__family_size    0.040
cat__pclass_1       0.033
num__sibsp          0.025
num__parch          0.024
cat__embarked_S     0.016
cat__pclass_2       0.015
cat__embarked_C     0.013
cat__is_alone_0     0.010
cat__is_alone_1     0.008
cat__embarked_Q     0.008
dtype: float64
